## Final dataset preparation
- Now, ones we have done cleaning and feature engineering, let's perform some final tasks :
1. Remove outliers
2. Encode categorical variables (if any)
3. standarize / normalize the data
4. Remove unnecessary columns for feeding into models

In [48]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Important thorughout this project
%matplotlib inline


In [49]:
df = pd.read_csv('../data/03_engineered/satellites_engineered.csv')

In [50]:
df.head()

,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,NORAD_CAT_ID,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,ORBIT_PERIOD_SEC,SEMI_MAJOR_AXIS,ORBIT_HEIGHT,PERIGEE,APOGEE,ORBITAL_SPEED,AGE_SINCE_LAUNCH,SAT_TYPE
0,2025-11-24 00:38:47.626368,13.763307,0.002630,90.2213,67.0218,189.7579,232.6051,900,4320,0.000861,8.510000e-06,0.0,6277.560908,7354.997884,983.997884,7335.654239,7374.341528,7.361687,61.898,C
1,2025-11-23 19:18:27.313056,13.528807,0.002055,90.2361,70.9479,102.8184,16.7196,902,82849,0.000074,5.800000e-07,0.0,6386.372286,7439.745563,1068.745563,7424.457630,7455.033496,7.319638,61.895,E
2,2025-11-23 16:55:04.202400,13.335800,0.007137,89.9895,212.6858,105.2414,309.9552,1512,93266,0.000132,7.500000e-07,0.0,6478.801456,7511.356523,1140.356523,7457.747972,7564.965075,7.284663,60.895,E
3,2025-11-23 22:01:14.414880,13.362341,0.006863,89.9085,124.4058,323.9637,161.4359,1520,93529,0.000215,1.200000e-06,0.0,6465.932675,7501.406738,1130.406738,7449.927584,7552.885891,7.289492,60.895,H
4,2025-11-23 23:24:19.305216,14.737562,0.000474,69.9169,179.8821,293.9595,66.1053,2826,3595,0.001312,8.522000e-05,0.0,5862.570613,7027.173125,656.173125,7023.839434,7030.506815,7.531445,58.895,A


----
### 1. Removing outliers
##### Key question : Why even remove outliers if we are going to perform anomaly detection?


### 1️⃣ Garbage-in → Garbage-out

* ML models (Isolation Forest, KMeans, etc.) assume that most of your data is “normal” to learn patterns.
* If your dataset accidentally has **obvious errors** (e.g., height = 0 km, speed = 1e6 m/s, negative values), the model may learn wrong patterns or consider normal data as anomalous.

---

### 2️⃣ Helps define “normal”

* Anomaly detection models define anomalies relative to what is normal.
* If there are **obvious outliers**, your baseline “normal” distribution will be skewed, reducing detection accuracy for true anomalies like orbital maneuvers.

---

### 3️⃣ Prevents false positives

* Spotting obvious errors ensures you **don’t flag bad data as anomalies**.
* Real anomalies should reflect **interesting satellite behavior**, not just bad CSV entries or measurement glitches.

---

✅ **Summary:**

* Spotting anomalies early is about **data quality**, not about defeating your anomaly detection goal.
* Once the dataset is clean, your ML models can focus on **real, subtle anomalies** (like unusual delta in orbital height, speed, or maneuvers).




---

## ❗For prototyping purpose, we will not perform outlier removal step for now, later on we may perform that!

### 2. Encode categorical variables 

In [51]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12603 entries, 0 to 12602
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   EPOCH              12603 non-null  object 
 1   MEAN_MOTION        12603 non-null  float64
 2   ECCENTRICITY       12603 non-null  float64
 3   INCLINATION        12603 non-null  float64
 4   RA_OF_ASC_NODE     12603 non-null  float64
 5   ARG_OF_PERICENTER  12603 non-null  float64
 6   MEAN_ANOMALY       12603 non-null  float64
 7   NORAD_CAT_ID       12603 non-null  int64  
 8   REV_AT_EPOCH       12603 non-null  int64  
 9   BSTAR              12603 non-null  float64
 10  MEAN_MOTION_DOT    12603 non-null  float64
 11  MEAN_MOTION_DDOT   12603 non-null  float64
 12  ORBIT_PERIOD_SEC   12603 non-null  float64
 13  SEMI_MAJOR_AXIS    12603 non-null  float64
 14  ORBIT_HEIGHT       12603 non-null  float64
 15  PERIGEE            12603 non-null  float64
 16  APOGEE             126

- We only have one categorical feature, that's going to be feed into the models and that is 'satellite type'.
- So we will use OneHotEncoder to encode the feature

#### ❗Removing the columns not requiring scaling

In [52]:
df = df.drop(columns=['EPOCH', 'NORAD_CAT_ID'])

In [53]:
from sklearn.preprocessing import OneHotEncoder

In [54]:
# Fit and transform
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_encoded = encoder.fit_transform(df[['SAT_TYPE']])

# Get actual category names
encoded_cols = encoder.get_feature_names_out(['SAT_TYPE'])

# Convert to DataFrame with proper column names
df_encoded = pd.concat([df.drop('SAT_TYPE', axis=1), pd.DataFrame(X_encoded, columns=encoded_cols)], axis=1)


In [55]:
df_encoded.head()

,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,...,SAT_TYPE_Q,SAT_TYPE_R,SAT_TYPE_S,SAT_TYPE_T,SAT_TYPE_U,SAT_TYPE_V,SAT_TYPE_W,SAT_TYPE_X,SAT_TYPE_Y,SAT_TYPE_Z
0,13.763307,0.002630,90.2213,67.0218,189.7579,232.6051,4320,0.000861,8.510000e-06,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,13.528807,0.002055,90.2361,70.9479,102.8184,16.7196,82849,0.000074,5.800000e-07,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,13.335800,0.007137,89.9895,212.6858,105.2414,309.9552,93266,0.000132,7.500000e-07,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,13.362341,0.006863,89.9085,124.4058,323.9637,161.4359,93529,0.000215,1.200000e-06,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,14.737562,0.000474,69.9169,179.8821,293.9595,66.1053,3595,0.001312,8.522000e-05,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [56]:
# We combined Main dataframe (excluding the SAT_TYPE) + (X encoded (values) + encoded_cols feature names) 
X_encoded

array([[0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.]], shape=(12603, 24))

In [57]:
encoded_cols

array(['SAT_TYPE_A', 'SAT_TYPE_B', 'SAT_TYPE_C', 'SAT_TYPE_D',
       'SAT_TYPE_E', 'SAT_TYPE_F', 'SAT_TYPE_G', 'SAT_TYPE_H',
       'SAT_TYPE_J', 'SAT_TYPE_K', 'SAT_TYPE_L', 'SAT_TYPE_M',
       'SAT_TYPE_N', 'SAT_TYPE_P', 'SAT_TYPE_Q', 'SAT_TYPE_R',
       'SAT_TYPE_S', 'SAT_TYPE_T', 'SAT_TYPE_U', 'SAT_TYPE_V',
       'SAT_TYPE_W', 'SAT_TYPE_X', 'SAT_TYPE_Y', 'SAT_TYPE_Z'],
      dtype=object)

In [58]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12603 entries, 0 to 12602
Data columns (total 41 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   MEAN_MOTION        12603 non-null  float64
 1   ECCENTRICITY       12603 non-null  float64
 2   INCLINATION        12603 non-null  float64
 3   RA_OF_ASC_NODE     12603 non-null  float64
 4   ARG_OF_PERICENTER  12603 non-null  float64
 5   MEAN_ANOMALY       12603 non-null  float64
 6   REV_AT_EPOCH       12603 non-null  int64  
 7   BSTAR              12603 non-null  float64
 8   MEAN_MOTION_DOT    12603 non-null  float64
 9   MEAN_MOTION_DDOT   12603 non-null  float64
 10  ORBIT_PERIOD_SEC   12603 non-null  float64
 11  SEMI_MAJOR_AXIS    12603 non-null  float64
 12  ORBIT_HEIGHT       12603 non-null  float64
 13  PERIGEE            12603 non-null  float64
 14  APOGEE             12603 non-null  float64
 15  ORBITAL_SPEED      12603 non-null  float64
 16  AGE_SINCE_LAUNCH   126

---
### 3. Feature Scaling 

- **Numeric Features:**  
  Features like `SEMI_MAJOR_AXIS`, `ORBIT_HEIGHT`, `ORBITAL_SPEED`, etc., have different units and ranges. Scaling them using **StandardScaler** standardizes the values (mean=0, std=1) so that all numeric features contribute equally to distance-based algorithms like **KMeans** and **Isolation Forest**.

- **Categorical Features (One-Hot Encoded):**  
  Features like `SAT_TYPE` are converted into binary columns (0/1). These **do not require scaling**, as their values are already normalized and scaling would distort the categorical meaning.

- **Key Idea:**  
  - Scale numeric continuous features.  
  - Keep one-hot categorical features as-is.  
  This ensures the model correctly interprets distances and patterns without bias from different units.


In [59]:
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer # lets you apply different preprocessing to different columns

In [60]:
scaler = StandardScaler()

encoded_cols = [col for col in df_encoded.columns if col.startswith('SAT_TYPE_')]
numeric_cols = df_encoded.drop(columns = encoded_cols).columns

ct = ColumnTransformer([
    ('scaler', StandardScaler(), numeric_cols), # scale numeric feature
    ('pass', 'passthrough', encoded_cols) # keep encoded columns as is
])

df_scaled = ct.fit_transform(df_encoded)

In [61]:
df_scaled

array([[-2.02208971,  0.71478887,  1.35517805, ...,  0.        ,
         0.        ,  0.        ],
       [-2.4010025 ,  0.51931518,  1.35589312, ...,  0.        ,
         0.        ,  0.        ],
       [-2.7128701 ,  2.24669624,  1.34397849, ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 0.22546517,  0.17887622, -0.50127133, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.22139269,  0.20508213, -0.50122301, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.47629379,  0.05030408, -0.97374951, ...,  0.        ,
         0.        ,  0.        ]], shape=(12603, 41))

In [62]:
# convert to dataframe                        # numeric_col is a series, so convert to a list 
df_scaled = pd.DataFrame(df_scaled, columns = numeric_cols.tolist() + encoded_cols)

In [63]:
df_scaled.head()

,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT,...,SAT_TYPE_Q,SAT_TYPE_R,SAT_TYPE_S,SAT_TYPE_T,SAT_TYPE_U,SAT_TYPE_V,SAT_TYPE_W,SAT_TYPE_X,SAT_TYPE_Y,SAT_TYPE_Z
0,-2.022090,0.714789,1.355178,-0.942297,0.320206,0.359963,-0.689055,0.173321,-0.015310,-0.045187,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-2.401002,0.519315,1.355893,-0.905716,-0.624997,-1.974070,5.214405,-0.003116,-0.018490,-0.045187,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-2.712870,2.246696,1.343978,0.414903,-0.598654,1.196229,5.997508,0.009780,-0.018422,-0.045187,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,-2.669983,2.153429,1.340065,-0.407631,1.779286,-0.409479,6.017279,0.028440,-0.018241,-0.045187,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,-0.447857,-0.017889,0.374159,0.109260,1.453082,-1.440140,-0.743557,0.274271,0.015453,-0.045187,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


- Save as model ready dataset

In [64]:
df_scaled.to_csv('../data/04_scaled/satellites_scaled.csv', index=False)

- Save the encoder also (for later use in isolation forest)

In [65]:
import joblib

joblib.dump(encoder, '../data/04_scaled/encoder_sat_type.joblib')

['../data/04_scaled/encoder_sat_type.joblib']

#### Now we are ready for Machine Learning models!